In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

In [2]:
corpus = [
    "burger was not bad",
    "fries were tasty",
    "service was amazing",
    "burger was not good",
    "fries were not tasty",
    "service was not amazing",
    "burger looked good but tasted awful",
    "fries looked bad but tasted great",
    "great another cold burger",
    "amazing service but terrible food"
]
labels = [
    "positive",
    "positive",
    "positive",
    "negative",
    "negative",
    "negative",
    "negative",
    "positive",
    "negative",
    "negative"
]

In [8]:
# --- BAG OF WORDS + N-GRAMS ---
# ngram_range=(1,2) znači:
# 1 = unigrami (pojedinačne riječi)
# 2 = bigrami (parovi riječi)
# ovo je parametar s kojim se igramo i pokušavamo povećati točnost
# kombinacije su (1, 1), (1, 2), (2, 2), (1, 3), (2, 3) itd.
vectorizer = CountVectorizer(ngram_range=(3, 4))
X = vectorizer.fit_transform(corpus)
# --- ML MODEL ---
model = MultinomialNB()
model.fit(X, labels)
# --- TEST REČENICA ---
new_text = ["food was not bad"]
# pretvaramo tekst u BoW / n-gram reprezentaciju
new_X = vectorizer.transform(new_text)
# predikcija
prediction = model.predict(new_X)
print("Prediction:", prediction[0])
# da vidiš koje featuree model koristi
print("\nVocabulary:")
print(vectorizer.get_feature_names_out())

Prediction: positive

Vocabulary:
['amazing service but' 'amazing service but terrible'
 'another cold burger' 'bad but tasted' 'bad but tasted great'
 'burger looked good' 'burger looked good but' 'burger was not'
 'burger was not bad' 'burger was not good' 'but tasted awful'
 'but tasted great' 'but terrible food' 'fries looked bad'
 'fries looked bad but' 'fries were not' 'fries were not tasty'
 'fries were tasty' 'good but tasted' 'good but tasted awful'
 'great another cold' 'great another cold burger' 'looked bad but'
 'looked bad but tasted' 'looked good but' 'looked good but tasted'
 'service but terrible' 'service but terrible food' 'service was amazing'
 'service was not' 'service was not amazing' 'was not amazing'
 'was not bad' 'was not good' 'were not tasty']


In [10]:
import pandas as pd

In [11]:
putanja = '/content/news.csv'
news = pd.read_csv (putanja)
pd.set_option("display.max_colwidth", None)
news.head(20)

,text,category
0,The prime minister met with regional leaders to discuss energy policy and border security.,politics
1,Parliament approved a new education reform after a long debate among coalition members.,politics
2,The president called for national unity during a speech on foreign policy and defense.,politics
3,Lawmakers introduced a bill aimed at reducing taxes for small municipalities.,politics
4,The opposition party criticized the government over healthcare funding and public spending.,politics
5,"A diplomatic meeting focused on trade relations, migration, and regional stability.",politics
6,The senate voted on judicial reforms and changes to electoral rules.,politics
7,Government officials announced a new housing program for young families.,politics
8,The mayor defended the city budget during a tense council session.,politics
9,"Ministers discussed climate policy, public transport, and energy prices.",politics


In [12]:
news.value_counts("category")

,count
category,
business,12
politics,12
sports,12
technology,12


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    news["text"],
    news["category"],
    test_size=0.25,
    random_state=42,
    stratify=news["category"]
)
# Bag of Words
vectorizer = CountVectorizer(stop_words="english")
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)
# model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_bow, y_train)
# predictions
y_pred = model.predict(X_test_bow)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.5833333333333334

Classification report:

              precision    recall  f1-score   support

    business       0.50      0.67      0.57         3
    politics       1.00      0.33      0.50         3
      sports       0.50      0.33      0.40         3
  technology       0.60      1.00      0.75         3

    accuracy                           0.58        12
   macro avg       0.65      0.58      0.56        12
weighted avg       0.65      0.58      0.56        12



In [14]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
# Bag of Words + n-grams
vectorizer_ng = CountVectorizer(stop_words="english", ngram_range=(1,2))
X_train_ng = vectorizer_ng.fit_transform(X_train)
X_test_ng = vectorizer_ng.transform(X_test)
model_ng = LogisticRegression(max_iter=1000)
model_ng.fit(X_train_ng, y_train)
y_pred_ng = model_ng.predict(X_test_ng)
print("Accuracy with n-grams:", accuracy_score(y_test, y_pred_ng))
print("\nClassification report with n-grams:\n")
print(classification_report(y_test, y_pred_ng))

Accuracy with n-grams: 0.6666666666666666

Classification report with n-grams:

              precision    recall  f1-score   support

    business       0.50      0.67      0.57         3
    politics       1.00      0.33      0.50         3
      sports       0.50      0.67      0.57         3
  technology       1.00      1.00      1.00         3

    accuracy                           0.67        12
   macro avg       0.75      0.67      0.66        12
weighted avg       0.75      0.67      0.66        12



In [15]:
print("Number of features with simple BoW:", X_train_bow.shape[1])
print("Number of features with n-grams:", X_train_ng.shape[1])

Number of features with simple BoW: 243
Number of features with n-grams: 482


In [16]:
new_articles = [
    "The minister discussed tax reform and foreign policy during a parliamentary debate.",
    "The striker scored twice and the team won the match in the final minutes.",
    "The company released a software patch to fix a serious security issue.",
    "The bank reported stronger earnings and lower operating costs this quarter."
]
new_X = vectorizer_ng.transform(new_articles)
predictions = model_ng.predict(new_X)
for article, pred in zip(new_articles, predictions):
    print("TEXT:", article)
    print("PREDICTED CATEGORY:", pred)
    print("-" * 60)

TEXT: The minister discussed tax reform and foreign policy during a parliamentary debate.
PREDICTED CATEGORY: politics
------------------------------------------------------------
TEXT: The striker scored twice and the team won the match in the final minutes.
PREDICTED CATEGORY: sports
------------------------------------------------------------
TEXT: The company released a software patch to fix a serious security issue.
PREDICTED CATEGORY: technology
------------------------------------------------------------
TEXT: The bank reported stronger earnings and lower operating costs this quarter.
PREDICTED CATEGORY: business
------------------------------------------------------------


**TF & IDF**

In [17]:
# --- LIBRARIES ---
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
# --- PODACI ---
corpus = [
    "burger was good",
    "fries were tasty",
    "service was amazing",
    "burger was not good",
    "fries were not tasty",
    "service was not amazing",
    "burger looked good but tasted awful",
    "fries looked bad but tasted great",
    "great another cold burger",
    "amazing service but terrible food"
]
labels = [
    "positive",
    "positive",
    "positive",
    "negative",
    "negative",
    "negative",
    "negative",
    "positive",
    "negative",
    "negative"
]

In [18]:
# --- TRAIN / TEST SPLIT ---
X_train, X_test, y_train, y_test = train_test_split(
    corpus,
    labels,
    test_size=0.3,
    random_state=42
)
# --- TF-IDF ---
vectorizer = TfidfVectorizer(ngram_range=(1,2), stop_words="english")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
# --- MODEL ---
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)
# --- PREDIKCIJE ---
y_pred = model.predict(X_test_tfidf)
# --- REZULTATI ---
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.3333333333333333

Classification report:

              precision    recall  f1-score   support

    negative       0.50      0.50      0.50         2
    positive       0.00      0.00      0.00         1

    accuracy                           0.33         3
   macro avg       0.25      0.25      0.25         3
weighted avg       0.33      0.33      0.33         3



In [19]:
# --- NOVI "TVITOVI" ZA TEST ---
new_texts = [
    "burger was amazing",
    "burger was not good",
    "fries were not bad",
    "service was terrible",
    "great burger but cold fries",
    "amazing service and tasty food"
]
# --- PRETVARANJE U TF-IDF ---
new_X = vectorizer.transform(new_texts)
# --- PREDIKCIJE ---
predictions = model.predict(new_X)
# --- ISPIS ---
for text, pred in zip(new_texts, predictions):
    print("TEXT:", text)
    print("PREDICTION:", pred)
    print("-" * 40)

TEXT: burger was amazing
PREDICTION: negative
----------------------------------------
TEXT: burger was not good
PREDICTION: negative
----------------------------------------
TEXT: fries were not bad
PREDICTION: negative
----------------------------------------
TEXT: service was terrible
PREDICTION: negative
----------------------------------------
TEXT: great burger but cold fries
PREDICTION: negative
----------------------------------------
TEXT: amazing service and tasty food
PREDICTION: negative
----------------------------------------
